# Level 3: Discrete Hodge Decomposition, Inflation Proofs, and Quotient Linear Programming

**Paper Package 002** — Pedagogical Progression: Level 3 of 5

**Target Audience:** Graduate students in theoretical physics, applied mathematics, or computational topology.

**Core Idea:** The Euclidean Hodge projection on graphs is an $\ell^2$-orthogonal split that distorts $\ell^1$ metrics. On unweighted cycle graphs $C_n$, the operator norm inflation is provably bounded by $R_{\mathrm{split}}^{\max}(C_n) = 3 - 2/n$. True $\ell^1$ quotient residuals require linear programming with strong duality.

*Evidence scope: finite toy-model demonstration only.*

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Locate and import code directory dynamically
cwd = Path.cwd().resolve()
for candidate in (cwd, cwd / "code", cwd.parent / "code", *(d / "code" for d in cwd.parents)):
    if (candidate / "constraint_closure_l1_structure.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from hodge_split_c3 import operator_norm_bound_cn, operator_norm_proof_cn, compute_weighted_triangle_phi
from regge_susy_saturation_solver import compute_primitive_difference_quotient
print('[OK] Level 3 environment initialized!')

## 1. Theorem: Sharp Operator Norm Bound on Cycle Graphs $C_n$

On the unweighted real cycle $C_n$, let the harmonic projection be:
$$P \omega = \left(\frac{\sum_{j=1}^n \omega_j}{n}\right) \mathbf{1}.$$

The split map $T \omega = ((I - P)\omega, P\omega)$ has output norm $\|T\omega\|_1 = \|(I - P)\omega\|_1 + \|P\omega\|_1$.

On standard basis vectors $e_j$, $(Pe_j)_k = 1/n$ for all $k$, so:
$$\|P e_j\|_1 = 1, \qquad \|(I - P)e_j\|_1 = \left(1 - \frac{1}{n}\right) + (n - 1)\left(\frac{1}{n}\right) = 2 - \frac{2}{n}.$$

Therefore, on every basis vector:
$$\|T e_j\|_1 = \left(2 - \frac{2}{n}\right) + 1 = 3 - \frac{2}{n}.$$

By linearity and the triangle inequality:
$$\|T \omega\|_1 \le \sum_j |\omega_j| \|T e_j\|_1 = \left(3 - \frac{2}{n}\right) \|\omega\|_1.$$

Since equality is achieved on any basis vector $e_j$, we have the exact operator norm:
$$\boxed{R_{\mathrm{split}}^{\max}(C_n) = 3 - \frac{2}{n}}.$$

Let's test this analytical bound across cycles $n = 3, 4, 5, 6, 10$:

In [ ]:
cycle_sizes = [3, 4, 5, 6, 10, 20]
for n in cycle_sizes:
    proof = operator_norm_proof_cn(n)
    bound = operator_norm_bound_cn(n)
    formula_str = f"3 - 2/{n} = {3.0 - 2.0/n:.4f}"
    print(f"C_{n:2d}: Ceiling = {bound:.6f} ({formula_str})")
    assert np.isclose(bound, 3.0 - 2.0 / n)
    assert proof["basis_attains_equality"] is True

# For C3, R_split = 3 - 2/3 = 7/3
assert np.isclose(operator_norm_bound_cn(3), 7.0 / 3.0)
print("[OK] Exact cycle ceiling R_split^max(Cn) = 3 - 2/n verified across all n!")

## 2. Weighted Triangles and Repair Concentration

What happens when edges carry non-uniform weights $a_i > 0$?
The weighted $\ell^1$ quotient norm is:
$$\Phi_a([\omega]) = \inf_\alpha \sum_{i=0}^2 a_i |\omega_i - (d_0 \alpha)_i|.$$

By the triangle inequality on cycles:
$$\Phi_a([\omega]) = \left(\min_{i} a_i\right) \left|\sum_{i=0}^2 \omega_i\right|.$$

The optimal $\ell^1$ repair concentrates the entire topological discrepancy on the single cheapest edge, leaving the other edges completely clean! In contrast, $\ell^2$ Hodge projections spread the harmonic flux uniformly regardless of weights.

In [ ]:
weights = [1.0, 5.0, 10.0]  # Edge 0 is much cheaper than edge 2
omega = [1.0, 1.0, 1.0]     # Net circulation = 3.0

phi_a = compute_weighted_triangle_phi(omega, weights)
cheapest_edge = int(np.argmin(weights))
circulation = float(np.sum(omega))

print(f"Edge weights:        {weights}")
print(f"Total circulation:   {circulation}")
print(f"Weighted Phi_a:      {phi_a:.6f}")
print(f"Optimal edge chosen: {cheapest_edge}")

# Expected: min(a) * |circulation| = 1.0 * 3.0 = 3.0
assert np.isclose(phi_a, 3.0)
print("[OK] Weighted triangle concentration verified!")

## 3. Primal-Dual Linear Programming for $\Phi_1([\omega])$

The general $\ell^1$ quotient problem is formulated as a Linear Program:
$$\min_{\alpha, t} \sum_{i=1}^m t_i \quad \text{subject to} \quad -t_i \le \omega_i - (d_0 \alpha)_i \le t_i, \quad t_i \ge 0.$$

The Dual Linear Program is:
$$\max_{w} \langle w, \omega \rangle \quad \text{subject to} \quad \|w\|_\infty \le 1, \quad d_0^T w = 0.$$

By LP Strong Duality, when a feasible solution exists, the primal minimum equals the dual maximum.

Let's test both an exact cochain $\omega_{\mathrm{exact}} = (1, -2, 1)$ and an obstructed circulation $\omega_{\mathrm{circ}} = (1, 1, 1)$:

In [ ]:
d0 = np.array([[-1.0, 1.0, 0.0], [0.0, -1.0, 1.0], [1.0, 0.0, -1.0]])

# 1. Exact cochain (Phi1 = 0)
res_exact = compute_primitive_difference_quotient([1.0, -2.0, 1.0], d0)
print("Exact cochain:")
print(f"  Optimization status: {res_exact['optimization_status']}")
print(f"  Closure status:      {res_exact['closure_status']}")
print(f"  Min L1 residual:     {res_exact['min_l1']:.6f}")
assert res_exact["is_optimal"]
assert np.isclose(res_exact["min_l1"], 0.0)

# 2. Obstructed cochain (Phi1 = 3)
res_obs = compute_primitive_difference_quotient([1.0, 1.0, 1.0], d0)
print("\nObstructed cochain:")
print(f"  Optimization status: {res_obs['optimization_status']}")
print(f"  Closure status:      {res_obs['closure_status']}")
print(f"  Min L1 residual:     {res_obs['min_l1']:.6f}")
assert res_obs["is_optimal"]
assert np.isclose(res_obs["min_l1"], 3.0)

print("\n[OK] Linear programming correctly distinguishes optimization success from vanishing residual!")